<a href="https://colab.research.google.com/github/Edomario082909/AHHHHR/blob/main/sdxl_v1.0_comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import threading, subprocess, re, time

# =========================
# 🔒 GESTIONE TOKEN
# =========================
MY_CIVITAI_TOKEN = "a84327c4d857915fd6730f0f0b637aac"

# =========================
# 🔧 SETUP SISTEMA
# =========================
!rm -f /etc/apt/sources.list.d/r2u.list
!apt -y update -qq && apt -y install -qq aria2 ffmpeg libsox-dev
!wget -q https://github.com/camenduru/gperftools/releases/download/v1.0/libtcmalloc_minimal.so.4 -O /content/libtcmalloc_minimal.so.4
%env LD_PRELOAD=/content/libtcmalloc_minimal.so.4

# =========================
# 📦 INSTALLAZIONE COMFYUI
# =========================
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI

# Installazione dipendenze base
!pip install -q torchsde omegaconf diffusers accelerate
!pip install -q -r requirements.txt

# ==========================================
# 🎵 AUDIO & EXTRA
# ==========================================
!pip install -q lucent taming-transformers-rom1504
!rm -rf /content/ComfyUI/comfy_api_nodes

# =========================
# 🧩 CUSTOM NODES
# =========================
%cd /content/ComfyUI/custom_nodes

!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!git clone https://github.com/city96/ComfyUI-GGUF.git
!pip install -q gguf
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git

# =========================
# 📥 DOWNLOAD MODELLI
# =========================
%cd /content/ComfyUI

# LISTA RINNOVATA CON I 20 URN FORNITI
my_list = [
    # ---- Flux1 ----
    ("2626167", "flux1_checkpoint.safetensors", "checkpoints"),
    ("729352",  "flux1_lora.safetensors",       "loras"),

    # ---- SDXL ----
    ("1811054", "sdxl_lora.safetensors",        "loras"),

    # ---- WanVideo (Text-to-Video) ----
    ("2057171", "wan_t2v_checkpoint.safetensors", "checkpoints"),
    ("2303105", "wan_t2v_lora_1.safetensors",   "loras"),
    ("2303113", "wan_t2v_lora_2.safetensors",   "loras"),

    # ---- WanVideo (Image-to-Video) ----
    ("2729619", "wan_i2v_lora_1.safetensors",   "loras"), # Indicato come 'unknown', di solito va nei loras
    ("2073605", "wan_i2v_lora_2.safetensors",   "loras"),
    ("2083303", "wan_i2v_lora_3.safetensors",   "loras"),
    ("2376136", "wan_i2v_lora_4.safetensors",   "loras"),
    ("2448064", "wan_i2v_lora_5.safetensors",   "loras"),
    ("2460386", "wan_i2v_lora_6.safetensors",   "loras"),
    ("2553151", "wan_i2v_lora_7.safetensors",   "loras"),
    ("2376143", "wan_i2v_lora_8.safetensors",   "loras"),
    ("2553271", "wan_i2v_lora_9.safetensors",   "loras"),

    # ---- ZimageTurbo ----
    ("2442439", "zimage_checkpoint.safetensors", "checkpoints"),
    ("2855359", "zimage_lora_1.safetensors",     "loras"),

    # ---- LTX Video ----
    ("2772932", "ltx_lora_1.safetensors",        "loras"),
    ("2803342", "ltx_lora_2.safetensors",        "loras"),
    ("2848299", "ltx_lora_3.safetensors",        "loras")
]

def download_model(v_id, name, folder):
    # Pausa di 2 secondi per evitare blocchi (Rate Limit) da Civitai
    time.sleep(2)

    url = f"https://civitai.com/api/download/models/{v_id}?token={MY_CIVITAI_TOKEN}"
    print(f"⬇️ In download: {name}")

    # Eseguiamo aria2c catturando gli errori
    cmd = ['aria2c', '-c', '-x', '16', '-s', '16', '-k', '1M', url, '-d', f"/content/ComfyUI/models/{folder}", '-o', name]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        print(f"✅ OK: {name}")
    else:
        print(f"❌ Errore/Skip: {name}")
        # Filtra l'output per stampare il motivo reale dell'errore (404, 401, ecc.)
        error_lines = [line.strip() for line in result.stderr.splitlines() if "HTTP response" in line or "error" in line.lower()]
        if error_lines:
            print("   Motivo:", error_lines[-1])
        else:
            print("   Motivo: ID Modello invalido o richiedente autorizzazione web.")

        # Elimina le tracce del file vuoto/corrotto
        os.system(f'rm -f "/content/ComfyUI/models/{folder}/{name}" "/content/ComfyUI/models/{folder}/{name}.aria2"')

for v_id, name, folder in my_list:
    download_model(v_id, name, folder)

# =========================
# 🌐 TUNNEL CLOUDFLARE
# =========================
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared-linux-amd64
!chmod 777 /content/cloudflared-linux-amd64

def tunnel():
    p = subprocess.Popen(
        ['/content/cloudflared-linux-amd64', 'tunnel', '--url', 'http://127.0.0.1:8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    for line in iter(p.stdout.readline, b''):
        line = line.decode('utf-8', errors='ignore')
        link = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if link:
            print(f"\n🟢 LINK COMFYUI: {link.group(0)}\n")

threading.Thread(target=tunnel, daemon=True).start()

# =========================
# ▶️ AVVIO
# =========================
!python main.py --dont-print-server --listen 0.0.0.0


